# **Implementing a Matrix Factorization-based Recommender System**

## **Represent user and item by Matrix Factorization**
 - Users and items are represented through matrix factorization.
  - A user-item interaction matrix $( R \in \mathbb{R}^{n \times m})$ is approximated as the product of two matrices: $( R \approx P \times Q)$, where $( P \in \mathbb{R}^{n \times d})$ and $( Q \in \mathbb{R}^{m \times d})$.
  - $ n $ is the number of users, $ m $ is the number of items, and $ d $ is the dimension of the embedding vectors.

**How to do Matrix Factorization**:
   - The goal is to find a good representation for users and items.
   - The objective is to minimize the differences between the predicted and actual interaction values: $ \min_{P,Q} \sum_{(u,i) \in R'} (r_{ui} - P_u Q_i)^2 $.
   - Not all elements in $ R $ are known; $ R' $ is the set of known elements in $ R $.
   - $ r_{ui} $ is the interaction record of user $ u $ and item $ i $.
   - $ P_u $ is the embedding vector for user $ u $, and $ Q_i $ is the embedding vector for item $ i $.
   - The interaction probability between user $ u $ and item $ i $ is $ r_{ui} = P_u Q_i $.

## **Requirements:**
In this practice, you will implement a recommender system using **Matrix Factorization**.
You should:
   - Construct a matrix factorization-based recommender system using the positive data `train_pos.npy` provided in project 3.
   - For each user-item pair $ u, i $ in `train_pos.npy`, $ R_{ui} = 1 $.
   - If a user-item pair $ u^*, i^* $ is not in `train_pos.npy`, $ R_{u^*i^*} = 0 $.
   - The task is to find a good embedding representation for each user and item.


## **Reference Workflow**:
   1. Load the data and construct an interaction matrix.
   2. Obtain the embedding representation for each user and item.
      - **Use the objective function above and optimize the embeddings via gradient descent.**
      - **Note: The number of negative samples is much larger than that of positive samples. You can sample some negative samples in each iteration instead of using all negative samples.**
   3. Validate the effectiveness of the model.

### **Deadline:** 5.20



## **1 Load and Explore the Dataset**

In [53]:
import numpy as np
import random

# Load the dataset
train_pos = np.load("train_pos.npy")  # Contains user-item pairs
users, items = set(train_pos[:, 0]), set(train_pos[:, 1])
len(train_pos),train_pos[:5],train_pos[-5:]

(26638,
 array([[   0, 1113,    1],
        [   0,  736,    1],
        [   0,  888,    1],
        [   0,  636,    1],
        [   1,  374,    1]], dtype=int64),
 array([[6014,  934,    1],
        [6014, 1960,    1],
        [6014,  937,    1],
        [6014, 1963,    1],
        [6014, 1485,    1]], dtype=int64))

In [54]:
n_user, n_item = max(users) + 1, max(items) + 1
n_user, n_item

(6015, 2347)

In [55]:
# --- Dataset Split: 8 : 1 : 1 ---
np.random.seed(42)
indices = np.random.permutation(len(train_pos))

n_train = int(len(train_pos) * 0.8)
n_val = int(len(train_pos) * 0.1)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_data = train_pos[train_idx]
val_data = train_pos[val_idx]
test_data = train_pos[test_idx]

print(f"Total samples: {len(train_pos)}")
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Build lookup sets for each split
train_set = set((int(u), int(i)) for u, i, _ in train_data)
val_set = set((int(u), int(i)) for u, i, _ in val_data)
test_set = set((int(u), int(i)) for u, i, _ in test_data)
all_pos_set = train_set | val_set | test_set

Total samples: 26638
Train: 21310, Val: 2663, Test: 2665


## **2. Initialize Parameters**

Initialize the embedding matrices $P$ for users and $Q$ for items. These matrices represent the user and item embeddings.

**Fill in the missing parts:**


In [56]:
# Define the embedding dimension
dim = 20

# Initialize user and item embeddings with small random values
P = np.random.normal(0, 0.01, (n_user, dim))  # User embeddings
Q = np.random.normal(0, 0.01, (n_item, dim))  # Item embeddings

## **3. Optimize the embeddings via gradient descent**

The loss function to optimize is Mean Squared Error (MSE):
$$
\text{Loss} = \sum_{(u, i) \in R'} (r_{ui} - P_u Q_i^T)^2
$$

OR add the regularization term:

$$
\text{Loss} = \sum_{(u, i) \in R'} (r_{ui} - P_u Q_i^T)^2 + \lambda (\|P_u\|^2 + \|Q_i\|^2)
$$

Here:
- $ R' $ is the set of the known elements in the $ R $
- $ r_{ui} $ is 1 for positive samples and 0 for negative samples.
- $ \lambda $ is the regularization term to prevent overfitting.


In [57]:
# Hyperparameters
learning_rate = 0.05
n_epochs = 40
lambda_reg = 0.1
neg_samples = 1  # number of negative samples per positive sample

# Training loop (only use train_data)
for epoch in range(n_epochs):
    total_loss = 0
    
    # Iterate over training positive samples only
    for user, item, _ in train_data:
        user, item = int(user), int(item)
        
        # --- Positive sample update ---
        pred = np.dot(P[user], Q[item])
        error = 1 - pred  # label = 1 for observed interactions
        
        P[user] += learning_rate * (error * Q[item] - lambda_reg * P[user])
        Q[item] += learning_rate * (error * P[user] - lambda_reg * Q[item])
        
        total_loss += error**2 + lambda_reg * (np.linalg.norm(P[user])**2 + np.linalg.norm(Q[item])**2)
        
        # --- Negative sample updates ---
        for _ in range(neg_samples):
            neg_item = random.randint(0, n_item - 1)
            # avoid sampling a positive item in TRAINING set
            while (user, neg_item) in train_set:
                neg_item = random.randint(0, n_item - 1)
            
            pred_neg = np.dot(P[user], Q[neg_item])
            error_neg = 0 - pred_neg  # label = 0 for unobserved interactions
            
            P[user] += learning_rate * (error_neg * Q[neg_item] - lambda_reg * P[user])
            Q[neg_item] += learning_rate * (error_neg * P[user] - lambda_reg * Q[neg_item])
            
            total_loss += error_neg**2 + lambda_reg * (np.linalg.norm(P[user])**2 + np.linalg.norm(Q[neg_item])**2)
    
    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {total_loss:.4f}")

Epoch 1/40, Loss: 21325.6295
Epoch 2/40, Loss: 21318.5588
Epoch 3/40, Loss: 21311.6407
Epoch 4/40, Loss: 21297.7517
Epoch 5/40, Loss: 21252.5904
Epoch 6/40, Loss: 21076.5530
Epoch 7/40, Loss: 20471.4949
Epoch 8/40, Loss: 19135.4916
Epoch 9/40, Loss: 17433.4755
Epoch 10/40, Loss: 15862.2061
Epoch 11/40, Loss: 14643.3327
Epoch 12/40, Loss: 13708.5520
Epoch 13/40, Loss: 13066.8692
Epoch 14/40, Loss: 12558.0854
Epoch 15/40, Loss: 12203.0755
Epoch 16/40, Loss: 11951.9417
Epoch 17/40, Loss: 11706.3947
Epoch 18/40, Loss: 11547.8801
Epoch 19/40, Loss: 11396.2077
Epoch 20/40, Loss: 11255.0192
Epoch 21/40, Loss: 11113.7164
Epoch 22/40, Loss: 11007.3432
Epoch 23/40, Loss: 10860.4051
Epoch 24/40, Loss: 10776.6185
Epoch 25/40, Loss: 10670.8772
Epoch 26/40, Loss: 10584.2534
Epoch 27/40, Loss: 10477.1883
Epoch 28/40, Loss: 10374.0853
Epoch 29/40, Loss: 10305.8385
Epoch 30/40, Loss: 10260.0818
Epoch 31/40, Loss: 10151.0394
Epoch 32/40, Loss: 10101.1527
Epoch 33/40, Loss: 10052.6940
Epoch 34/40, Loss: 

## **4 Verification**

Choose an appropriate metric to evaluate the results.

In [58]:
def evaluate(data, pos_set, n_neg=5000, n_auc_pos=2000, n_auc_neg=2000):
    """Evaluate on a given dataset split."""
    # Positive accuracy
    pos_correct = 0
    for user, item, _ in data:
        pred = np.dot(P[int(user)], Q[int(item)])
        if pred > 0.5:
            pos_correct += 1
    pos_acc = pos_correct / len(data) if len(data) > 0 else 0
    
    # Negative accuracy (sample negatives not in ALL positive sets)
    neg_correct = 0
    for _ in range(n_neg):
        u = random.randint(0, n_user - 1)
        i = random.randint(0, n_item - 1)
        while (u, i) in all_pos_set:
            i = random.randint(0, n_item - 1)
        pred = np.dot(P[u], Q[i])
        if pred <= 0.5:
            neg_correct += 1
    neg_acc = neg_correct / n_neg
    
    # Sample MSE
    mse_total = 0
    count = 0
    for user, item, _ in data[:min(len(data), 5000)]:
        pred = np.dot(P[int(user)], Q[int(item)])
        mse_total += (1 - pred)**2
        count += 1
    for _ in range(count):
        u = random.randint(0, n_user - 1)
        i = random.randint(0, n_item - 1)
        while (u, i) in all_pos_set:
            i = random.randint(0, n_item - 1)
        pred = np.dot(P[u], Q[i])
        mse_total += (0 - pred)**2
    mse = mse_total / (count * 2) if count > 0 else 0
    
    # AUC
    pos_preds = []
    indices = np.random.choice(len(data), min(n_auc_pos, len(data)), replace=False)
    for idx in indices:
        user, item, _ = data[idx]
        pos_preds.append(np.dot(P[int(user)], Q[int(item)]))
    
    neg_preds = []
    cnt = 0
    while cnt < n_auc_neg:
        u = random.randint(0, n_user - 1)
        i = random.randint(0, n_item - 1)
        if (u, i) not in all_pos_set:
            neg_preds.append(np.dot(P[u], Q[i]))
            cnt += 1
    
    pos_preds = np.array(pos_preds)
    neg_preds = np.array(neg_preds)
    auc = np.mean(pos_preds[:, None] > neg_preds[None, :])
    
    return pos_acc, neg_acc, mse, auc

# Evaluate on all splits
print("=" * 50)
train_pos_acc, train_neg_acc, train_mse, train_auc = evaluate(train_data, train_set)
print(f"Train  -- Pos Acc: {train_pos_acc:.4f}, Neg Acc: {train_neg_acc:.4f}, MSE: {train_mse:.6f}, AUC: {train_auc:.4f}")

val_pos_acc, val_neg_acc, val_mse, val_auc = evaluate(val_data, val_set)
print(f"Val    -- Pos Acc: {val_pos_acc:.4f}, Neg Acc: {val_neg_acc:.4f}, MSE: {val_mse:.6f}, AUC: {val_auc:.4f}")

test_pos_acc, test_neg_acc, test_mse, test_auc = evaluate(test_data, test_set)
print(f"Test   -- Pos Acc: {test_pos_acc:.4f}, Neg Acc: {test_neg_acc:.4f}, MSE: {test_mse:.6f}, AUC: {test_auc:.4f}")
print("=" * 50)

Train  -- Pos Acc: 0.8084, Neg Acc: 0.9432, MSE: 0.095016, AUC: 0.9660
Val    -- Pos Acc: 0.6181, Neg Acc: 0.9406, MSE: 0.169730, AUC: 0.8671
Test   -- Pos Acc: 0.6206, Neg Acc: 0.9458, MSE: 0.169313, AUC: 0.8668


In [59]:
# --- AUC Calculation ---
# AUC measures the probability that a randomly chosen positive sample
# is ranked higher than a randomly chosen negative sample.

n_auc_samples = 5000

# Sample positive predictions
pos_indices = np.random.choice(len(train_pos), n_auc_samples, replace=False)
pos_preds = []
for idx in pos_indices:
    user, item, _ = train_pos[idx]
    pos_preds.append(np.dot(P[int(user)], Q[int(item)]))

# Sample negative predictions
neg_preds = []
count = 0
while count < n_auc_samples:
    u = random.randint(0, n_user - 1)
    i = random.randint(0, n_item - 1)
    if (u, i) not in train_set:
        neg_preds.append(np.dot(P[u], Q[i]))
        count += 1

pos_preds = np.array(pos_preds)
neg_preds = np.array(neg_preds)

# Compute AUC: count how many (pos, neg) pairs are ranked correctly
auc = 0
for pp in pos_preds:
    auc += np.sum(pp > neg_preds)
auc /= (len(pos_preds) * len(neg_preds))

print(f"AUC: {auc:.4f}")

AUC: 0.9476
